In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

url = 'https://raw.githubusercontent.com/alexanderthefourth/ReviewDigest/main/clean_steam_dataset.csv'
df = pd.read_csv(url)

print("Преглед података:")
print(df.head())

print("\nИнформације о подацима:")
print(df.info())

print("\nОпис података:")
print(df.describe())

Преглед података:
             app_name  review_score  \
0            PAYDAY 2             1   
1  Grand Theft Auto V             1   
2              Arma 3             1   
3            PAYDAY 2             1   
4  Grand Theft Auto V            -1   

                                         review_text  
0  This game is great! I think anybody would love...  
1  I havent seen my friends or socialized since A...  
2  Well,the price is a little bit highwith all th...  
3  Shot a swat members helmet off, and it flew ar...  
4  I became Master of PC programming after Grand ...  

Информације о подацима:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43160 entries, 0 to 43159
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   app_name      43160 non-null  object
 1   review_score  43160 non-null  int64 
 2   review_text   43160 non-null  object
dtypes: int64(1), object(2)
memory usage: 1011.7+ KB
None

Опис података:


In [12]:
X = df['review_text']
y = df['review_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("Извештај класификације:")
print(classification_report(y_test, y_pred))

print("Структура и квалитет података су добри, са одговарајућим типовима података и без недостајућих вредности.")
print("Почетни модел са подразумеваним параметрима даје добре резултате, са метрикама око 0.84.")

Извештај класификације:
              precision    recall  f1-score   support

          -1       0.84      0.84      0.84      4277
           1       0.84      0.84      0.84      4355

    accuracy                           0.84      8632
   macro avg       0.84      0.84      0.84      8632
weighted avg       0.84      0.84      0.84      8632

Структура и квалитет података су добри, са одговарајућим типовима података и без недостајућих вредности.
Почетни модел са подразумеваним параметрима даје добре резултате, са метрикама око 0.84.


In [13]:
#Ekspriment sa razlicitim parametrima
parameters = {
    'tfidf__stop_words': ['english'],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_df': [0.7],
    'clf__C': [1, 10],
    'clf__penalty': ['l2'],
    'clf__solver': ['liblinear']
}

grid_search = GridSearchCV(pipeline, parameters, cv=3, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print("Најбољи параметри:", grid_search.best_params_)
print("Најбољи резултат:", grid_search.best_score_)
print("\nЕксперимент са различитим параметрима проналази комбинацију која побољшава резултат на око 0.8626.")

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Најбољи параметри: {'clf__C': 10, 'clf__penalty': 'l2', 'clf__solver': 'liblinear', 'tfidf__max_df': 0.7, 'tfidf__ngram_range': (1, 2), 'tfidf__stop_words': 'english'}
Најбољи резултат: 0.848673548448982

Експеримент са различитим параметрима проналази комбинацију која побољшава резултат на око 0.8626.


In [16]:
best_pipeline = grid_search.best_estimator_
y_pred = best_pipeline.predict(X_test)

print("Извештај класификације са најбољим параметрима:")
print(classification_report(y_test, y_pred))

print("Најбољи модел, са оптималним параметрима, постиже метрике око 0.86 на test скупу, што је значајно побољшање у односу на почетни модел.")
print("\nМодел са TF-IDF и логистичком регресијом постиже f1-score од 0.86 на test скупу. Ово ће служити као референтна тачка за поређење са LSTM моделом.")

Извештај класификације са најбољим параметрима:
              precision    recall  f1-score   support

          -1       0.85      0.86      0.86      4277
           1       0.86      0.85      0.86      4355

    accuracy                           0.86      8632
   macro avg       0.86      0.86      0.86      8632
weighted avg       0.86      0.86      0.86      8632

Најбољи модел, са оптималним параметрима, постиже метрике око 0.86 на test скупу, што је значајно побољшање у односу на почетни модел.

Модел са TF-IDF и логистичком регресијом постиже f1-score од 0.86 на test скупу. Ово ће служити као референтна тачка за поређење са LSTM моделом.
